In [ ]:
import sqlite3
from pathlib import Path
from subprocess import run

from crossfiledialog import choose_folder

from src.tag import AutoTagger

### **📂 Open a (empty) folder as collection**

In [ ]:
collection_path = Path(choose_folder())
collection_path

### **🖼️ Add some images if needed**

In [ ]:
# Select a folder, import all images within it
image_folder_path = Path(choose_folder())
run(["target/release/cli", str(collection_path), "add", "folder", str(image_folder_path)], check=False)

### **🍵 Prepare tagger and database**

In [ ]:
tagger = AutoTagger()
index_db = sqlite3.connect(collection_path / "index.db")
index_db.row_factory = sqlite3.Row

### **🔍 Select ids and paths**

In [ ]:
images = index_db.execute("SELECT id, hash, fullname FROM images").fetchall()
images = [{
    "id": img["id"],
    "path": collection_path / "images" / img["hash"] / img["fullname"],
    "thumbnail_path": collection_path / "images" / img["hash"] / "thumbnail.png"
} for img in images]
images

### **🏷️ Tag it!**

In [ ]:
from IPython.display import Image, display

for image in images:
    tags = tagger.tag(image["path"])
    display(Image(filename=image["thumbnail_path"], width=96))
    print(f"Tags: {tags}")